# `encode_params` — Direct Parameter Specification

The most robust entry point. The user provides a typed constructor and vector length `N`.
No code is parsed and no vector is ever materialised.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from qiskit.quantum_info import Statevector

from pyencode import (
    encode_params,
    DISCRETE, UNIFORM, STEP, SQUARE,
    SINE, COSINE,
    MULTI_DISCRETE, MULTI_SINE,
)

## DISCRETE — single nonzero entry

$f_i = P\,\delta_{ik}$ &nbsp;&nbsp; Complexity: $\mathcal{O}(m)$

In [ ]:
circuit, info = encode_params(DISCRETE(k=3, P=5.0), N=8)
print(info)
circuit.draw('mpl')

In [ ]:
# Verify: statevector should have all weight at index 3
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))

In [ ]:
# DISCRETE at index 0 (no X gates needed)
circuit, info = encode_params(DISCRETE(k=0, P=1.0), N=8)
print(f"Gate count: {info.gate_count}")
circuit.draw('mpl')

In [ ]:
# DISCRETE at last index
circuit, info = encode_params(DISCRETE(k=7, P=1.0), N=8)
print(f"Gate count: {info.gate_count}  (all 3 bits set -> 3 X gates)")
circuit.draw('mpl')

## UNIFORM — constant vector

$f_i = c$ &nbsp;&nbsp; Complexity: $\mathcal{O}(m)$ ($m$ Hadamard gates)

In [ ]:
circuit, info = encode_params(UNIFORM(c=1.0), N=8)
print(info)
circuit.draw('mpl')

In [ ]:
# Verify: all amplitudes equal
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))

In [ ]:
# Larger N: still just m Hadamard gates
circuit, info = encode_params(UNIFORM(c=3.0), N=32)
print(f"N=32, m=5: gate count = {info.gate_count}")
circuit.draw('mpl')

## STEP — prefix constant, rest zero

$f_i = c\,\mathbf{1}[i < k_s]$ &nbsp;&nbsp; Complexity: $\mathcal{O}(m)$

In [ ]:
circuit, info = encode_params(STEP(k_s=4, c=1.0), N=8)
print(info)
circuit.draw('mpl')

In [ ]:
# Verify: first 4 entries nonzero, last 4 zero
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))

In [ ]:
# Step with non-power-of-2 boundary
circuit, info = encode_params(STEP(k_s=3, c=1.0), N=8)
print(f"k_s=3: gate count = {info.gate_count}")
circuit.draw('mpl')

## SQUARE — contiguous block

$f_i = c\,\mathbf{1}[k_1 \leq i < k_2]$ &nbsp;&nbsp; Complexity: $\mathcal{O}(m)$

In [ ]:
circuit, info = encode_params(SQUARE(k1=2, k2=6, c=1.0), N=8)
print(info)
circuit.draw('mpl')

In [ ]:
# Verify: entries 2,3,4,5 nonzero
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))

In [ ]:
# Aligned square: f[8:16] on N=16
circuit, info = encode_params(SQUARE(k1=8, k2=16, c=1.0), N=16)
print(f"Aligned square: gate count = {info.gate_count}")
circuit.draw('mpl')

## SINE — sinusoidal vector via QFT

$f_i = A\sin(2\pi n i/N + \varphi)$ &nbsp;&nbsp; Complexity: $\mathcal{O}(m^2)$

In [ ]:
circuit, info = encode_params(SINE(n=1, A=1.0), N=8)
print(info)
circuit.draw('mpl')

In [ ]:
# Verify against expected sine vector
sv = np.abs(np.array(Statevector(circuit)))
k = np.arange(8)
expected = np.sin(2 * np.pi * 1 * k / 8)
expected = np.abs(expected / np.linalg.norm(expected))
print("Circuit:  ", np.round(sv, 4))
print("Expected: ", np.round(expected, 4))

In [ ]:
# Higher mode with phase
circuit, info = encode_params(SINE(n=2, A=1.0, phi=np.pi/4), N=16)
print(f"n=2, phi=pi/4, N=16: gate count = {info.gate_count}")
circuit.draw('mpl')

## COSINE — cosine vector via QFT

$f_i = A\cos(2\pi n i/N + \varphi)$ &nbsp;&nbsp; Complexity: $\mathcal{O}(m^2)$

In [ ]:
circuit, info = encode_params(COSINE(n=1, A=1.0), N=8)
print(info)
circuit.draw('mpl')

In [ ]:
# Verify
sv = np.abs(np.array(Statevector(circuit)))
k = np.arange(8)
expected = np.cos(2 * np.pi * 1 * k / 8)
expected = np.abs(expected / np.linalg.norm(expected))
print("Circuit:  ", np.round(sv, 4))
print("Expected: ", np.round(expected, 4))

## MULTI_DISCRETE — multiple point loads

$f_i = \sum_j P_j\,\delta_{i,k_j}$ &nbsp;&nbsp; Complexity: $\mathcal{O}(|S|\,m)$

In [ ]:
circuit, info = encode_params(
    MULTI_DISCRETE(vectors=[DISCRETE(k=1, P=3.0), DISCRETE(k=6, P=4.0)]),
    N=8,
)
print(info)
circuit.draw('mpl')

In [ ]:
# Verify: weight at indices 1 and 6 only
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))

In [ ]:
# 5 point loads
circuit, info = encode_params(
    MULTI_DISCRETE(vectors=[
        DISCRETE(k=1, P=1.0), DISCRETE(k=5, P=2.0),
        DISCRETE(k=10, P=3.0), DISCRETE(k=20, P=1.5),
        DISCRETE(k=30, P=0.5),
    ]),
    N=32,
)
print(f"5 loads, N=32: gate count = {info.gate_count}")

## MULTI_SINE — sum of sinusoidal modes

$f_i = \sum_t A_t \sin(2\pi n_t i/N)$ &nbsp;&nbsp; Complexity: $\mathcal{O}(m^2)$

In [ ]:
circuit, info = encode_params(
    MULTI_SINE(modes=[SINE(n=1, A=2.0), SINE(n=3, A=1.0)]),
    N=16,
)
print(info)
circuit.draw('mpl')

## Composite via list syntax

Pass a list of constructors. Same-type lists auto-promote to the multi-type.

In [ ]:
# List of DISCRETE -> MULTI_DISCRETE
circuit, info = encode_params(
    [DISCRETE(k=1, P=3.0), DISCRETE(k=6, P=4.0)],
    N=8,
)
print(f"List of DISCRETE -> {info.vector_type}, gates={info.gate_count}")

In [ ]:
# List of SINE -> MULTI_SINE
circuit, info = encode_params(
    [SINE(n=1, A=2.0), SINE(n=3, A=1.0)],
    N=16,
)
print(f"List of SINE -> {info.vector_type}, gates={info.gate_count}")

## Validation and circuit code

In [ ]:
# validate=True simulates the circuit and checks correctness
circuit, info = encode_params(
    SINE(n=1, A=1.0), N=8,
    validate=True, tol=1e-6,
)
print(f"Validated: {info.validated}")

In [ ]:
# circuit_code: standalone Python/Qiskit script
_, info = encode_params(DISCRETE(k=3, P=1.0), N=8)
print(info.circuit_code)